# LeetCode 792 — Number of Matching Subsequences bằng Python

Link đề: https://leetcode.com/problems/number-of-matching-subsequences/ · Độ khó: **Medium** · Chủ đề: Trie, Array, Hash Table, String, Binary Search, Dynamic Programming, Sorting

Skeleton LeetCode:

```python
class Solution:
    def numMatchingSubseq(self, s: str, words: List[str]) -> int:
        ...
```

Ghi chú: bài có **đáp án duy nhất** (một số nguyên), nên so sánh bằng `==`. Notebook đối chiếu 4 cách giải với hai oracle độc lập (kiểm tra dãy con bằng iterator và liệt kê **mọi** dãy con của `s` ngắn) trên mọi chuỗi nhỏ và hàng nghìn input ngẫu nhiên có seed.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: đếm số từ trong `words` là **dãy con** (subsequence) của `s` — tức xoá một số ký tự (có thể không xoá) mà **giữ nguyên thứ tự** các ký tự còn lại. Từ trùng nhau được **đếm nhiều lần**.
2. Nắm các công cụ Python: hai con trỏ, `str.find(sub, start)`, `bisect_left`, danh sách các vị trí theo ký tự, `ord(c) - 97` để đổi chữ thành chỉ số, `collections.defaultdict`/`Counter`, iterator `iter(s)` với toán tử `in`.
3. Đi từ **mỗi từ quét `s` bằng hai con trỏ** → **danh sách vị trí + nhị phân** → **"xếp hàng chờ theo chữ cái" (bucket) quét `s` đúng một lần** → **bảng "vị trí kế tiếp" `next[c][i]`** (nhảy `O(1)`).
4. Hiểu mấu chốt: kiểm tra từ `w` là dãy con của `s` là **tham lam**: mỗi ký tự của `w` khớp với lần xuất hiện **sớm nhất** sau vị trí vừa khớp. Với nhiều từ, ta tránh quét lại `s` bằng cách *nhảy* (nhị phân / bảng) hoặc *dùng chung một lần quét* (bucket).
5. Tránh các lỗi kinh điển: nhầm dãy con với **chuỗi con** (`w in s`), quên bắt đầu tìm **sau** vị trí vừa khớp, sửa danh sách trong lúc duyệt, bỏ trùng bằng `set` (mất bội số), và lệch `bisect_left`/`bisect_right`.

## 2. Đề bài & ví dụ

Cho chuỗi `s` và mảng chuỗi `words`. Trả về **số** phần tử `words[i]` là **dãy con** của `s`.

**Dãy con** của một chuỗi là chuỗi mới tạo từ chuỗi gốc bằng cách xoá một số ký tự (có thể không xoá) mà không đổi thứ tự tương đối của các ký tự còn lại. Ví dụ `"ace"` là dãy con của `"abcde"`.

**Ví dụ 1:** `s = "abcde"`, `words = ["a","bb","acd","ace"]` → `3` (các từ `"a"`, `"acd"`, `"ace"`; còn `"bb"` thì `s` chỉ có một chữ `b`).

**Ví dụ 2:** `s = "dsahjpjauf"`, `words = ["ahjpjau","ja","ahbwzgqnuk","tnmlanowax"]` → `2` (`"ahjpjau"` và `"ja"`).

**Ràng buộc:**

- `1 <= s.length <= 5 * 10^4`
- `1 <= words.length <= 5000`
- `1 <= words[i].length <= 50`
- `s` và `words[i]` chỉ gồm chữ cái thường tiếng Anh.

Tổng độ dài các từ tối đa `5000 × 50 = 2,5 × 10^5`; còn `s` dài `5 × 10^4`. Quét `s` cho từng từ sẽ tốn `5000 × 5·10^4 = 2,5 × 10^8` bước — quá chậm với Python.

In [1]:
import random
import time
from bisect import bisect_left
from collections import Counter, defaultdict, deque
from itertools import product
from typing import List

ex1 = ("abcde", ["a", "bb", "acd", "ace"])
ex2 = ("dsahjpjauf", ["ahjpjau", "ja", "ahbwzgqnuk", "tnmlanowax"])
for s, words in (ex1, ex2):
    print(f"s = {s!r} (dài {len(s)}), {len(words)} từ: {words}")

s = 'abcde' (dài 5), 4 từ: ['a', 'bb', 'acd', 'ace']
s = 'dsahjpjauf' (dài 10), 4 từ: ['ahjpjau', 'ja', 'ahbwzgqnuk', 'tnmlanowax']


## 3. Kiến thức nền

### 3.1 Dãy con và chuỗi con

**Chuỗi con (substring)** là các ký tự **liên tiếp**; **dãy con (subsequence)** có thể **bỏ ký tự ở giữa**. `"ace"` là dãy con của `"abcde"` nhưng **không** là chuỗi con (`"ace" in "abcde"` là `False`). Toán tử `in` của Python trên chuỗi kiểm tra chuỗi con, không phải dãy con.

In [2]:
s = "abcde"
print("'ace' in s    :", "ace" in s, "(chuỗi con: sai)")
print("'bcd' in s    :", "bcd" in s, "(chuỗi con: đúng)")
print("'ace' là dãy con của s ? Có: a(0) -> c(2) -> e(4), các chỉ số tăng dần")

'ace' in s    : False (chuỗi con: sai)
'bcd' in s    : True (chuỗi con: đúng)
'ace' là dãy con của s ? Có: a(0) -> c(2) -> e(4), các chỉ số tăng dần


### 3.2 Kiểm tra một từ có là dãy con: hai con trỏ (tham lam)

Duyệt `s` từ trái; giữ con trỏ `j` là ký tự **tiếp theo của `w` cần khớp**. Gặp `s[i] == w[j]` thì `j += 1`. Nếu `j` chạm `len(w)` thì `w` là dãy con. Cách này **đúng** nhờ tham lam: khớp càng sớm càng để lại nhiều chỗ cho phần sau.

In [3]:
def is_subseq(w, s):
    j = 0
    for ch in s:
        if j < len(w) and ch == w[j]:
            j += 1
    return j == len(w)


for w in ("a", "bb", "acd", "ace", "aec"):
    print(f"{w!r:6} là dãy con của {ex1[0]!r}:", is_subseq(w, ex1[0]))

'a'    là dãy con của 'abcde': True
'bb'   là dãy con của 'abcde': False
'acd'  là dãy con của 'abcde': True
'ace'  là dãy con của 'abcde': True
'aec'  là dãy con của 'abcde': False


### 3.3 `str.find(sub, start)`: tìm kể từ một vị trí

`s.find(c, start)` trả về chỉ số **đầu tiên** `>= start` của `c` trong `s`, hoặc `-1` nếu không có. Dùng nó để "nhảy" tới lần xuất hiện kế tiếp mà không cần vòng lặp Python, vì `find` chạy bằng C (nhanh).

In [4]:
s = "abcabc"
print(s.find("c"), s.find("c", 3), s.find("c", 6), s.find("z"))
# kiểm tra dãy con bằng find: mỗi ký tự tìm SAU vị trí vừa khớp
def is_subseq_find(w, s):
    pos = 0
    for ch in w:
        pos = s.find(ch, pos) + 1        # +1: lần sau bắt đầu SAU ký tự vừa khớp
        if pos == 0:                     # find trả -1 -> pos = 0
            return False
    return True


print(is_subseq_find("ace", "abcde"), is_subseq_find("bb", "abcde"), is_subseq_find("bb", "abcbd"))

2 5 -1 -1
True False True


### 3.4 Danh sách vị trí theo ký tự và `bisect_left`

Với mỗi chữ cái, lưu danh sách **chỉ số xuất hiện trong `s`** (đã sắp tăng). `bisect_left(lst, x)` trả chỉ số đầu tiên của `lst` có giá trị `>= x`; vậy phần tử `lst[bisect_left(lst, x)]` là lần xuất hiện **đầu tiên của chữ đó ở vị trí `>= x`** — thay cho `find`, nhưng mỗi lần tra là `O(log)`.

In [5]:
s = "abcabcab"
positions = defaultdict(list)
for i, ch in enumerate(s):
    positions[ch].append(i)
print(dict(positions))
lst = positions["a"]
for start in (0, 1, 3, 4, 7):
    k = bisect_left(lst, start)
    print(f"từ vị trí {start}: lần xuất hiện kế tiếp của 'a' =", lst[k] if k < len(lst) else None)

{'a': [0, 3, 6], 'b': [1, 4, 7], 'c': [2, 5]}
từ vị trí 0: lần xuất hiện kế tiếp của 'a' = 0
từ vị trí 1: lần xuất hiện kế tiếp của 'a' = 3
từ vị trí 3: lần xuất hiện kế tiếp của 'a' = 3
từ vị trí 4: lần xuất hiện kế tiếp của 'a' = 6
từ vị trí 7: lần xuất hiện kế tiếp của 'a' = None


### 3.5 Đổi chữ cái thành chỉ số: `ord(c) - ord('a')`

`ord('a') = 97`. `ord(c) - 97` cho `0..25`, dùng làm chỉ số mảng thay vì từ điển. Ta dùng khi muốn một bảng `26` hàng (một hàng mỗi chữ cái).

In [6]:
print([ord(c) - ord("a") for c in "abz"], [chr(97 + i) for i in (0, 1, 25)])
buckets = [[] for _ in range(26)]
buckets[ord("c") - 97].append("cat")
print({chr(97 + i): b for i, b in enumerate(buckets) if b})

[0, 1, 25] ['a', 'b', 'z']
{'c': ['cat']}


### 3.6 Đếm trùng bằng `Counter`

Đề cho phép từ **trùng nhau** và mỗi lần xuất hiện **đều được đếm**. `Counter(words)` cho bội số mỗi từ; nếu tối ưu bằng cách kiểm tra từng từ khác nhau một lần, phải **nhân với bội số** (dùng `set` sẽ làm mất bội số — lỗi hay gặp).

In [7]:
words = ["a", "a", "ace", "a"]
c = Counter(words)
print(c, "| số từ khác nhau:", len(set(words)), "| tổng số từ:", len(words))
s = "abcde"
print("đếm đúng (nhân bội số):", sum(is_subseq(w, s) * m for w, m in c.items()), "| đếm sai (set):", sum(is_subseq(w, s) for w in set(words)))

Counter({'a': 3, 'ace': 1}) | số từ khác nhau: 2 | tổng số từ: 4
đếm đúng (nhân bội số): 4 | đếm sai (set): 2


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 1: `s = "abcde"`, từ `"acd"`: tìm `a` ở vị trí 0; sau đó tìm `c` **bắt đầu từ vị trí 1** → thấy ở vị trí 2; rồi tìm `d` **từ vị trí 3** → vị trí 3. Thành công. Từ `"bb"`: `b` ở vị trí 1; tìm `b` **từ vị trí 2** → không còn → thất bại.

Tính tay mọi từ:

| từ | các bước khớp (vị trí trong `s`) | kết quả |
|---|---|---|
| `a` | `a@0` | có |
| `bb` | `b@1`, rồi không còn `b` sau vị trí 1 | không |
| `acd` | `a@0 → c@2 → d@3` | có |
| `ace` | `a@0 → c@2 → e@4` | có |

Đáp án `3`.

**Bẫy:** nếu mỗi lần tìm lại từ đầu `s` (quên "sau vị trí vừa khớp"), `"bb"` sẽ "khớp" hai lần cùng một chữ `b@1`. Mọi lời giải hợp lệ đều phải **tiến lên** trong `s`.

**Vấn đề hiệu năng:** `5000` từ × quét cả `s` dài `5·10^4` = `2,5·10^8` bước. Ta cần một cách để **không quét lại `s`**.

In [8]:
s, words = ex1
for w in words:
    pos, trace = 0, []
    for ch in w:
        found = s.find(ch, pos)
        if found == -1:
            trace.append(f"{ch}: không còn từ vị trí {pos}")
            break
        trace.append(f"{ch}@{found}")
        pos = found + 1
    print(f"{w!r:6}", " -> ".join(trace))

'a'    a@0
'bb'   b@1 -> b: không còn từ vị trí 2
'acd'  a@0 -> c@2 -> d@3
'ace'  a@0 -> c@2 -> e@4


## 5. Cách 1 — Brute Force: mỗi từ quét `s` bằng hai con trỏ

**Ý tưởng:** đúng theo định nghĩa: với mỗi từ, dùng hàm `is_subseq` (hai con trỏ ở mục 3.2) và đếm các từ trả `True`.

In [9]:
def count_brute(s, words):
    return sum(1 for w in words if is_subseq(w, s))


print(count_brute(*ex1), count_brute(*ex2))
assert count_brute(*ex1) == 3 and count_brute(*ex2) == 2

3 2


**Độ phức tạp:** mỗi từ quét `O(|s|)` nên tổng `O(|words| · |s|)` = `2,5·10^8` ở giới hạn lớn nhất (rất chậm trong Python). Bộ nhớ `O(1)`.

**Nút thắt:** ta đọc lại các chữ của `s` **không liên quan** tới từ đang xét. Với mỗi từ ta chỉ cần biết "lần xuất hiện kế tiếp của chữ `c` kể từ vị trí `p`".

**Cải tiến nhỏ bằng `find` (vẫn là ý tưởng brute):** mỗi ký tự của `w` dùng `s.find(c, pos)` chạy bằng C; tổng `O(Σ|w| · tìm-kiếm)` nhưng `find` vẫn có thể quét dài nên xấu nhất vẫn `O(|words|·|s|)` (với từ thưa chữ). Ta dùng bản này làm oracle nhanh cho đầu vào lớn.

In [10]:
def count_find(s, words):
    total = 0
    for w in words:
        pos = 0
        for ch in w:
            pos = s.find(ch, pos) + 1
            if pos == 0:
                break
        else:
            total += 1
    return total


print(count_find(*ex1), count_find(*ex2))
assert count_find(*ex1) == 3 and count_find(*ex2) == 2

3 2


## 6. Ý tưởng thuật toán / bước đột phá

Có ba ý tưởng để **không quét lại `s`**:

1. **Nhảy bằng danh sách vị trí + nhị phân:** lưu, với mỗi chữ cái, mảng các chỉ số xuất hiện (tăng dần). Với từ `w`, giữ `cur` = vị trí được phép bắt đầu; mỗi chữ `c` của `w` tra `lst[bisect_left(lst, cur)]` rồi `cur = vị trí đó + 1`. Mỗi ký tự `O(log |s|)`.
2. **Bảng `next[c][i]`:** tiền xử lý **một** lần: `next[c][i]` = chỉ số nhỏ nhất `>= i` có `s[·] == c` (hoặc `-1`). Mỗi ký tự của từ tra `O(1)`. Tổng `O(26·|s| + Σ|w|)`.
3. **Xếp hàng chờ theo chữ cái (bucket):** quét `s` **đúng một lần**. Mỗi từ "chờ" chữ cái tiếp theo mà nó cần; khi gặp chữ đó trong `s`, mọi từ đang chờ chữ này **tiến một bước** và chuyển sang hàng chờ của chữ kế tiếp (hoặc được tính là khớp nếu hết). Tổng `O(|s| + Σ|w|)` — mỗi ký tự của mỗi từ được xử lý đúng một lần.

Cả ba đều dựa vào tính tham lam: **khớp sớm nhất** là tối ưu.

## 7. Các cách tiếp theo

Ngoài Cách 1 (mục 5), ta có thêm 3 cách: **Cách 2** vị trí + nhị phân, **Cách 3** bucket (một lần quét `s`), **Cách 4** bảng `next`.

### Cách 2 — Danh sách vị trí + `bisect_left`

**Giải thích từng đoạn:**

1. `pos[c]` = danh sách các chỉ số của chữ `c` trong `s` (tăng dần).
2. Với mỗi từ `w`: `cur = 0`. Với mỗi `c` trong `w`: `lst = pos[c]`; `k = bisect_left(lst, cur)`; nếu `k == len(lst)` thì thất bại; ngược lại `cur = lst[k] + 1`.
3. Nếu qua hết các chữ của `w` mà không thất bại thì tính một lần.

In [11]:
def count_bisect(s, words):
    pos = defaultdict(list)
    for i, ch in enumerate(s):
        pos[ch].append(i)
    total = 0
    for w in words:
        cur = 0
        for ch in w:
            lst = pos.get(ch)
            if not lst:
                break
            k = bisect_left(lst, cur)
            if k == len(lst):
                break
            cur = lst[k] + 1
        else:
            total += 1
    return total


print(count_bisect(*ex1), count_bisect(*ex2))
assert count_bisect(*ex1) == 3 and count_bisect(*ex2) == 2

3 2


In [12]:
# Code nộp LeetCode cho Cách 2
from bisect import bisect_left
from collections import defaultdict


class SolutionBisect:
    def numMatchingSubseq(self, s: str, words: List[str]) -> int:
        pos = defaultdict(list)
        for i, ch in enumerate(s):
            pos[ch].append(i)
        ans = 0
        for w in words:
            cur = 0
            for ch in w:
                lst = pos.get(ch)
                if not lst:
                    break
                k = bisect_left(lst, cur)
                if k == len(lst):
                    break
                cur = lst[k] + 1
            else:
                ans += 1
        return ans

**Độ phức tạp:** xây `pos` `O(|s|)`; mỗi ký tự của mỗi từ `O(log |s|)`. Tổng `O(|s| + Σ|w| log |s|)`; bộ nhớ `O(|s|)`.

### Cách 3 — Bucket: xếp hàng chờ theo chữ cái, quét `s` một lần

**Giải thích từng đoạn:**

1. Với mỗi từ, ban đầu nó chờ chữ **đầu tiên** của nó: đặt `(từ, 0)` vào `buckets[chữ đầu]`.
2. Duyệt từng ký tự `ch` của `s`. Lấy ra **toàn bộ** hàng chờ của `ch` (`waiting = buckets[ch]`) và **làm rỗng** `buckets[ch]` trước khi xử lý (rất quan trọng, xem lỗi 3).
3. Với mỗi `(w, i)` trong `waiting`: nó vừa khớp ký tự thứ `i`; nếu `i + 1 == len(w)` thì hết từ → `ans += 1`; ngược lại đặt `(w, i + 1)` vào hàng chờ của `w[i + 1]`.
4. Trả `ans`.

In [13]:
def count_bucket(s, words):
    buckets = defaultdict(list)
    for w in words:
        buckets[w[0]].append((w, 0))
    ans = 0
    for ch in s:
        waiting = buckets.pop(ch, [])              # lấy ra và làm rỗng hàng chờ của ch
        for w, i in waiting:
            if i + 1 == len(w):
                ans += 1                            # đã khớp hết từ
            else:
                buckets[w[i + 1]].append((w, i + 1))
    return ans


print(count_bucket(*ex1), count_bucket(*ex2))
assert count_bucket(*ex1) == 3 and count_bucket(*ex2) == 2

3 2


In [14]:
# Code nộp LeetCode cho Cách 3
from collections import defaultdict


class SolutionBucket:
    def numMatchingSubseq(self, s: str, words: List[str]) -> int:
        buckets = defaultdict(list)
        for w in words:
            buckets[w[0]].append((w, 0))
        ans = 0
        for ch in s:
            waiting = buckets.pop(ch, [])
            for w, i in waiting:
                if i + 1 == len(w):
                    ans += 1
                else:
                    buckets[w[i + 1]].append((w, i + 1))
        return ans

**Độ phức tạp:** mỗi ký tự của `s` làm một thao tác `pop`; mỗi cặp `(từ, i)` được xử lý **đúng một lần** (vì khi chữ `w[i]` xuất hiện, nó bị chuyển hàng chờ). Tổng `O(|s| + Σ|w|)` = `~3·10^5` ở giới hạn lớn nhất; bộ nhớ `O(|words|)`.

### Cách 4 — Bảng "vị trí kế tiếp" `next[c][i]`

**Ý tưởng:** biến phép `find` thành tra bảng `O(1)`. Xây `nxt[c][i]` (với `i ∈ [0, |s|]`) = chỉ số nhỏ nhất `>= i` mà `s[·] == c`, hoặc `-1` (nhập `|s|` làm lính canh). Dựng **từ phải sang trái**: `nxt[c][|s|] = -1` mọi `c`; với `i` từ `|s|−1` về `0`: sao chép cột `i+1` rồi đặt `nxt[s[i]][i] = i`. Từ `w`: `cur = 0`; mỗi chữ `c`: `p = nxt[c][cur]`; nếu `p == -1` thất bại; ngược lại `cur = p + 1`.

In [15]:
def build_next(s):
    n = len(s)
    nxt = [[-1] * (n + 1) for _ in range(26)]
    for i in range(n - 1, -1, -1):
        for c in range(26):
            nxt[c][i] = nxt[c][i + 1]
        nxt[ord(s[i]) - 97][i] = i
    return nxt


def count_next(s, words):
    nxt = build_next(s)
    ans = 0
    for w in words:
        cur = 0
        for ch in w:
            p = nxt[ord(ch) - 97][cur]
            if p == -1:
                break
            cur = p + 1
        else:
            ans += 1
    return ans


nxt = build_next("abcde")
print("nxt['a'] =", nxt[0], "| nxt['c'] =", nxt[2], "| nxt['e'] =", nxt[4])
print(count_next(*ex1), count_next(*ex2))
assert count_next(*ex1) == 3 and count_next(*ex2) == 2

nxt['a'] = [0, -1, -1, -1, -1, -1] | nxt['c'] = [2, 2, 2, -1, -1, -1] | nxt['e'] = [4, 4, 4, 4, 4, -1]
3 2


In [16]:
# Code nộp LeetCode cho Cách 4
class SolutionNext:
    def numMatchingSubseq(self, s: str, words: List[str]) -> int:
        n = len(s)
        nxt = [[-1] * (n + 1) for _ in range(26)]
        for i in range(n - 1, -1, -1):
            for c in range(26):
                nxt[c][i] = nxt[c][i + 1]
            nxt[ord(s[i]) - 97][i] = i
        ans = 0
        for w in words:
            cur = 0
            for ch in w:
                p = nxt[ord(ch) - 97][cur]
                if p == -1:
                    break
                cur = p + 1
            else:
                ans += 1
        return ans

**Độ phức tạp:** xây bảng `O(26·|s|)` (≈ `1,3·10^6` ô), mỗi ký tự từ `O(1)`; tổng `O(26|s| + Σ|w|)`; bộ nhớ `O(26|s|)`. Hữu ích khi có **rất nhiều truy vấn** trên cùng một `s` (xây bảng một lần).

## 8. TRACE từng bước

**Trace Cách 3 (bucket)** với `s = "abcde"`, `words = ["a","bb","acd","ace"]`. Ký hiệu hàng chờ theo chữ cái: `buckets[x]` là các từ đang cần chữ `x` (kèm chỉ số đã khớp).

| bước | ký tự `ch` | lấy ra khỏi `buckets[ch]` | hành động | `ans` | hàng chờ sau bước |
|---|---|---|---|---|---|
| khởi tạo | — | — | mỗi từ vào hàng chờ chữ đầu | 0 | `a: [a, acd, ace]`, `b: [bb]` |
| 1 | `a` | `a, acd, ace` | `a` hết từ (+1); `acd` → chờ `c`; `ace` → chờ `c` | 1 | `b: [bb]`, `c: [acd, ace]` |
| 2 | `b` | `bb` | `bb` đã khớp `b` đầu, chờ `b` thứ hai | 1 | `b: [bb]`, `c: [acd, ace]` |
| 3 | `c` | `acd, ace` | `acd` → chờ `d`; `ace` → chờ `e` | 1 | `b: [bb]`, `d: [acd]`, `e: [ace]` |
| 4 | `d` | `acd` | hết từ (+1) | 2 | `b: [bb]`, `e: [ace]` |
| 5 | `e` | `ace` | hết từ (+1) | 3 | `b: [bb]` |

Kết thúc: `ans = 3`; `bb` còn chờ chữ `b` thứ hai (không bao giờ tới). Hàm trace in đúng bảng trên:

In [17]:
def trace_bucket(s, words):
    buckets = defaultdict(list)
    for w in words:
        buckets[w[0]].append((w, 0))

    def show():
        return ", ".join(f"{c}: {[w for w, _ in lst]}" for c, lst in sorted(buckets.items()) if lst)

    print(f"s = {s!r}, words = {words}")
    print("khởi tạo:", show())
    ans = 0
    for step, ch in enumerate(s, 1):
        waiting = buckets.pop(ch, [])
        acts = []
        for w, i in waiting:
            if i + 1 == len(w):
                ans += 1
                acts.append(f"{w}: hết từ (+1)")
            else:
                buckets[w[i + 1]].append((w, i + 1))
                acts.append(f"{w}: chờ {w[i + 1]!r}")
        print(f"bước {step}: ch={ch!r} | lấy {[w for w, _ in waiting] or '-'} | {'; '.join(acts) or '(không ai chờ)'} | ans={ans} | {show()}")
    print("=> đáp án", ans)
    return ans


assert trace_bucket(*ex1) == 3

s = 'abcde', words = ['a', 'bb', 'acd', 'ace']
khởi tạo: a: ['a', 'acd', 'ace'], b: ['bb']
bước 1: ch='a' | lấy ['a', 'acd', 'ace'] | a: hết từ (+1); acd: chờ 'c'; ace: chờ 'c' | ans=1 | b: ['bb'], c: ['acd', 'ace']
bước 2: ch='b' | lấy ['bb'] | bb: chờ 'b' | ans=1 | b: ['bb'], c: ['acd', 'ace']
bước 3: ch='c' | lấy ['acd', 'ace'] | acd: chờ 'd'; ace: chờ 'e' | ans=1 | b: ['bb'], d: ['acd'], e: ['ace']
bước 4: ch='d' | lấy ['acd'] | acd: hết từ (+1) | ans=2 | b: ['bb'], e: ['ace']
bước 5: ch='e' | lấy ['ace'] | ace: hết từ (+1) | ans=3 | b: ['bb']
=> đáp án 3


**Trace Cách 4 (bảng `next`)** cho từ `"ace"` trên `s = "abcde"`: `cur = 0`; `a`: `nxt[a][0] = 0` → `cur = 1`; `c`: `nxt[c][1] = 2` → `cur = 3`; `e`: `nxt[e][3] = 4` → `cur = 5`; qua hết → khớp. Với `"bb"`: `b`: `nxt[b][0] = 1` → `cur = 2`; `b`: `nxt[b][2] = -1` → thất bại. Hàm trace:

In [18]:
def trace_next(s, w):
    nxt = build_next(s)
    cur = 0
    print(f"từ {w!r} trên s = {s!r}")
    for ch in w:
        p = nxt[ord(ch) - 97][cur]
        print(f"  tìm {ch!r} từ vị trí {cur}: nxt = {p}", end="")
        if p == -1:
            print("  -> thất bại")
            return False
        cur = p + 1
        print(f"  -> khớp tại {p}, cur = {cur}")
    print("  -> khớp hết")
    return True


assert trace_next("abcde", "ace") is True
assert trace_next("abcde", "bb") is False

từ 'ace' trên s = 'abcde'
  tìm 'a' từ vị trí 0: nxt = 0  -> khớp tại 0, cur = 1
  tìm 'c' từ vị trí 1: nxt = 2  -> khớp tại 2, cur = 3
  tìm 'e' từ vị trí 3: nxt = 4  -> khớp tại 4, cur = 5
  -> khớp hết
từ 'bb' trên s = 'abcde'
  tìm 'b' từ vị trí 0: nxt = 1  -> khớp tại 1, cur = 2
  tìm 'b' từ vị trí 2: nxt = -1  -> thất bại


## 9. Code nộp LeetCode

Chọn **bucket**: một lần quét `s`, `O(|s| + Σ|w|)` thời gian và `O(|words|)` bộ nhớ phụ. Giữ nguyên `class Solution`, `numMatchingSubseq(self, s, words)`:

In [19]:
from collections import defaultdict
from typing import List


class Solution:
    def numMatchingSubseq(self, s: str, words: List[str]) -> int:
        buckets = defaultdict(list)            # buckets[c]: các từ đang chờ chữ c, kèm chỉ số đã khớp
        for w in words:
            buckets[w[0]].append((w, 0))
        ans = 0
        for ch in s:
            waiting = buckets.pop(ch, [])      # lấy ra và LÀM RỖNG hàng chờ trước khi xử lý
            for w, i in waiting:
                if i + 1 == len(w):            # đã khớp hết từ
                    ans += 1
                else:                          # chuyển sang chờ chữ kế tiếp
                    buckets[w[i + 1]].append((w, i + 1))
        return ans


print(Solution().numMatchingSubseq("abcde", ["a", "bb", "acd", "ace"]))                                    # 3
print(Solution().numMatchingSubseq("dsahjpjauf", ["ahjpjau", "ja", "ahbwzgqnuk", "tnmlanowax"]))           # 2
assert Solution().numMatchingSubseq(*ex1) == 3 and Solution().numMatchingSubseq(*ex2) == 2

3
2


## 10. Vì sao đúng

**Bước 1 — Tham lam kiểm tra dãy con.** Nếu `w` là dãy con của `s` thì khớp **sớm nhất** có thể cũng thành công. Chứng minh trao đổi: gọi `p_1 < p_2 < … < p_m` là một cách khớp bất kỳ, `q_1 < … < q_m` là cách khớp tham lam (`q_k` là vị trí đầu tiên `> q_{k−1}` có `s[·] = w[k]`). Quy nạp: `q_1 ≤ p_1` (vì `q_1` là lần xuất hiện đầu tiên); giả sử `q_{k−1} ≤ p_{k−1}`; thế thì `p_k > p_{k−1} ≥ q_{k−1}` và `s[p_k] = w[k]`, nên `p_k` là ứng viên của "vị trí đầu tiên `> q_{k−1}`" suy ra `q_k ≤ p_k`. Do đó tham lam khớp được mọi ký tự khi bất kỳ cách khớp nào khớp được.

**Bước 2 — Các cách giải thực hiện đúng thuật toán tham lam:**

- *Hai con trỏ / `find` / `bisect` / bảng `next`:* mỗi bước tìm lần xuất hiện đầu tiên **sau** vị trí vừa khớp (`cur` luôn là `vị trí khớp trước + 1`).
- *Bucket:* bất biến — sau khi xử lý `s[0..t]`, mỗi từ `w` đang ở hàng chờ `w[i+1]` với `i+1` = số ký tự khớp tham lam của `w` trên `s[0..t]`. Khi gặp `s[t+1] = ch`, các từ đang chờ `ch` **khớp tham lam** ký tự đó (vì `ch` là lần xuất hiện đầu tiên sau lần khớp trước của chúng), và các từ không chờ `ch` không đổi. Hàng chờ phải được **tách ra trước** khi xử lý để một từ vừa tiến bước không bị xử lý lại bởi **cùng** ký tự `s[t+1]` (vì mỗi ký tự của `s` chỉ khớp được một ký tự của mỗi từ).

**Bước 3 — Đếm trùng.** `words` có thể trùng; ta duyệt từng phần tử của `words` (hoặc nhân bội số) nên mỗi lần xuất hiện đều được tính đúng một lần.

Ta kiểm chứng bằng máy trên **mọi** chuỗi nhỏ (mục 12D) với oracle liệt kê toàn bộ dãy con.

## 11. Time / Space Complexity

Đặt `n = |s|`, `W = Σ|w|` (tổng độ dài các từ), `m = |words|`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Hai con trỏ mỗi từ | `O(m · n)` | `O(1)` | `2,5·10^8` ở giới hạn lớn: quá chậm |
| 2. Vị trí + nhị phân | `O(n + W log n)` | `O(n)` | đơn giản, nhanh |
| 3. Bucket | `O(n + W)` | `O(m)` | **đáp án chính**; một lần quét `s` |
| 4. Bảng `next` | `O(26n + W)` | `O(26n)` | nhiều truy vấn trên cùng `s` |

Đo thực tế ở giới hạn lớn nhất (`|s| = 5·10^4`, `5000` từ dài `≤ 50`, nhiều từ là dãy con của `s`):

In [20]:
def random_instance(rng, n, m, maxlen, alphabet, match_ratio=0.6):
    s = "".join(rng.choice(alphabet) for _ in range(n))
    words = []
    for _ in range(m):
        L = rng.randint(1, maxlen)
        if rng.random() < match_ratio and n >= L:
            idx = sorted(rng.sample(range(n), L))           # một dãy con chắc chắn của s
            words.append("".join(s[i] for i in idx))
        else:
            words.append("".join(rng.choice(alphabet) for _ in range(L)))
    return s, words


rng = random.Random(2024)
ALPHA = "abcdefghijklmnopqrstuvwxyz"
s_big, words_big = random_instance(rng, 50000, 5000, 50, ALPHA)
print("n =", len(s_big), "| số từ =", len(words_big), "| W =", sum(map(len, words_big)))


def timeit(fn, *args):
    t0 = time.perf_counter()
    r = fn(*args)
    return r, time.perf_counter() - t0


ref, dt = timeit(count_find, s_big, words_big)
print(f"find (oracle nhanh) : {ref} khớp, {dt * 1000:8.1f} ms")
for name, fn in [("bisect", count_bisect), ("bucket", count_bucket), ("next", count_next)]:
    r, dt = timeit(fn, s_big, words_big)
    assert r == ref
    print(f"{name:20s}: {r} khớp, {dt * 1000:8.1f} ms")
# brute hai con trỏ: chỉ chạy trên 100 từ để thấy tỉ lệ thời gian
r100, dt = timeit(count_brute, s_big, words_big[:100])
print(f"brute hai con trỏ, chỉ 100/5000 từ: {dt * 1000:8.1f} ms (-> khoảng {dt * 50:.1f} s cho 5000 từ)")

n = 50000 | số từ = 5000 | W = 127911


find (oracle nhanh) : 5000 khớp,      9.2 ms


bisect              : 5000 khớp,     32.6 ms


bucket              : 5000 khớp,     16.8 ms


next                : 5000 khớp,     84.2 ms


brute hai con trỏ, chỉ 100/5000 từ:    126.1 ms (-> khoảng 6.3 s cho 5000 từ)


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề, (B) edge case, (C) phản ví dụ: cài đặt sai phải bị bắt, (D) vét cạn mọi chuỗi nhỏ với oracle "liệt kê mọi dãy con", (E) ngẫu nhiên có seed, (F) giới hạn lớn nhất.

### A. Ví dụ của đề

In [21]:
IMPLS = {
    "brute": count_brute,
    "find": count_find,
    "bisect": count_bisect,
    "bucket": count_bucket,
    "next": count_next,
    "Solution": lambda s, w: Solution().numMatchingSubseq(s, list(w)),
    "SolutionBisect": lambda s, w: SolutionBisect().numMatchingSubseq(s, list(w)),
    "SolutionBucket": lambda s, w: SolutionBucket().numMatchingSubseq(s, list(w)),
    "SolutionNext": lambda s, w: SolutionNext().numMatchingSubseq(s, list(w)),
}


def oracle_iter(s, words):
    """Oracle 1: kiểm tra dãy con bằng iterator (`c in it` tiêu thụ iterator tới khi gặp c)."""
    total = 0
    for w in words:
        it = iter(s)
        total += all(c in it for c in w)
    return total


def all_subsequences(s):
    """Oracle 2: tập MỌI dãy con của s (chỉ dùng cho s ngắn)."""
    out = set()
    n = len(s)
    for mask in range(1 << n):
        out.add("".join(s[i] for i in range(n) if mask >> i & 1))
    return out


def oracle_enum(s, words):
    subs = all_subsequences(s)
    return sum(1 for w in words if w in subs)


def check(s, words, want=None, names=None):
    if want is None:
        want = oracle_iter(s, words)
    for name in (names or IMPLS):
        got = IMPLS[name](s, list(words))
        assert got == want, (name, s, words, got, want)


check(*ex1, 3)
check(*ex2, 2)
assert oracle_iter(*ex1) == oracle_enum(*ex1) == 3 and oracle_iter(*ex2) == oracle_enum(*ex2) == 2
print("ví dụ 1, 2: cả", len(IMPLS), "cài đặt và 2 oracle đều đúng")

ví dụ 1, 2: cả 9 cài đặt và 2 oracle đều đúng


### B. Edge case

- `s` một ký tự; từ một ký tự.
- Từ **dài hơn** `s` (không thể).
- Từ **bằng** `s`; từ rỗng không có (mọi từ dài `≥ 1`).
- Từ **trùng nhau** nhiều lần (phải đếm đủ).
- Từ chứa chữ **không có** trong `s`.
- `s` toàn một chữ cái (`"aaaa"`) và từ `"aa"`, `"aaaaa"`.
- Từ có chữ lặp (`"bb"`, `"abab"`) kiểm tra "tiến lên sau vị trí vừa khớp".

In [22]:
cases = [
    ("a", ["a"], 1), ("a", ["b"], 0), ("a", ["aa"], 0),
    ("abc", ["abc", "abcd", "cba"], 1),
    ("abc", ["a", "a", "a", "b", "c", "d"], 5),                 # trùng được đếm đủ
    ("aaaa", ["a", "aa", "aaa", "aaaa", "aaaaa"], 4),
    ("abcde", ["bb", "acd", "ace", "aec", "edcba"], 2),
    ("abab", ["abab", "aab", "bba", "bab"], 3),
    ("xyz", ["w", "wx", "z"], 1),
    ("a" * 50, ["a" * 50, "a" * 51, "a" * 49], 2),
]
for s, words, want in cases:
    check(s, words, want)
    assert oracle_iter(s, words) == want
print(len(cases), "edge case: tất cả cài đặt đúng")

10 edge case: tất cả cài đặt đúng


### C. Phản ví dụ: cài đặt sai phải bị bắt

In [23]:
def bug_substring(s, words):                       # nhầm dãy con với chuỗi con
    return sum(1 for w in words if w in s)


def bug_restart(s, words):                         # luôn tìm lại từ đầu s (quên "sau vị trí vừa khớp")
    total = 0
    for w in words:
        total += all(s.find(ch) != -1 for ch in w)
    return total


def bug_set(s, words):                             # bỏ trùng bằng set: mất bội số
    return sum(1 for w in set(words) if is_subseq(w, s))


def bug_bucket_same_list(s, words):                # duyệt hàng chờ trong khi chính nó bị thêm vào
    buckets = defaultdict(list)
    for w in words:
        buckets[w[0]].append((w, 0))
    ans = 0
    for ch in s:
        for w, i in buckets[ch]:                   # SAI: không tách hàng chờ ra trước
            if i + 1 == len(w):
                ans += 1
            else:
                buckets[w[i + 1]].append((w, i + 1))
        buckets[ch] = []                           # (dọn sau) — nhưng các từ chờ cùng chữ đã bị xử lý ngay trong vòng này
    return ans


def bug_bisect_right_cur(s, words):                # bisect_right thay cho bisect_left: bỏ qua đúng vị trí cur
    from bisect import bisect_right
    pos = defaultdict(list)
    for i, ch in enumerate(s):
        pos[ch].append(i)
    total = 0
    for w in words:
        cur = 0
        for ch in w:
            lst = pos.get(ch, [])
            k = bisect_right(lst, cur)
            if k == len(lst):
                break
            cur = lst[k] + 1
        else:
            total += 1
    return total


BUGS = [bug_substring, bug_restart, bug_set, bug_bucket_same_list, bug_bisect_right_cur]
rng = random.Random(1)
tests = []
for _ in range(600):
    n = rng.randint(1, 8)
    tests.append(("".join(rng.choice("abc") for _ in range(n)), ["".join(rng.choice("abc") for _ in range(rng.randint(1, 4))) for _ in range(rng.randint(1, 5))]))
for bug in BUGS:
    bad = next(((s, w) for s, w in tests if bug(s, list(w)) != oracle_iter(s, w)), None)
    assert bad is not None, bug.__name__
    print(f"{bug.__name__:22s} sai trên s={bad[0]!r}, words={bad[1]}: ra {bug(bad[0], list(bad[1]))}, đúng {oracle_iter(*bad)}")

bug_substring          sai trên s='bcbc', words=['bccc', 'b', 'cc']: ra 1, đúng 2
bug_restart            sai trên s='cab', words=['bbcb']: ra 1, đúng 0
bug_set                sai trên s='cabcb', words=['c', 'c']: ra 1, đúng 2
bug_bucket_same_list   sai trên s='bcabccc', words=['b', 'a', 'bbca']: ra 3, đúng 2
bug_bisect_right_cur   sai trên s='c', words=['b', 'bc', 'c', 'bb', 'ba']: ra 0, đúng 1


### D. Vét cạn mọi chuỗi nhỏ

Mọi chuỗi `s` trên bảng chữ `{a, b, c}` có độ dài `1..6` (`3 + 9 + 27 + 81 + 243 + 729 = 1092` chuỗi), cùng một mảng `words` gồm **mọi** từ trên `{a, b, c}` độ dài `1..3` (`39` từ) cộng thêm vài bản sao trùng. So sánh toàn bộ cài đặt với **oracle liệt kê mọi dãy con** (độc lập với tham lam).

In [24]:
all_words = ["".join(t) for L in (1, 2, 3) for t in product("abc", repeat=L)]
words_dup = all_words + ["a", "ab", "ab", "abc", "cba", "cba", "cba"]
cnt = 0
for L in range(1, 7):
    for t in product("abc", repeat=L):
        s = "".join(t)
        subs = all_subsequences(s)
        want = sum(1 for w in words_dup if w in subs)
        for name, fn in IMPLS.items():
            assert fn(s, list(words_dup)) == want, (name, s)
        # từng từ riêng lẻ cũng khớp
        for w in all_words:
            assert (w in subs) == is_subseq(w, s) == bool(count_bucket(s, [w]))
        cnt += 1
print(cnt, "chuỗi s vét cạn x", len(words_dup), "từ: 9 cài đặt khớp oracle liệt kê mọi dãy con")

1092 chuỗi s vét cạn x 46 từ: 9 cài đặt khớp oracle liệt kê mọi dãy con


### E. Ngẫu nhiên có seed, đúng ràng buộc

Chuỗi `s` dài tới 300, bảng chữ từ 1 đến 26 chữ cái, `words` tới 60 từ dài tới 12 (một nửa là dãy con của `s` chắc chắn). So sánh với `oracle_iter`.

In [25]:
rng = random.Random(20240792)
count = matches = total_words = 0
for trial in range(1500):
    k_alpha = rng.choice([1, 2, 3, 5, 10, 26])
    alphabet = ALPHA[:k_alpha]
    n = rng.randint(1, 300)
    m = rng.randint(1, 60)
    s, words = random_instance(rng, n, m, min(12, max(1, n)), alphabet, match_ratio=rng.choice([0.2, 0.5, 0.8]))
    want = oracle_iter(s, words)
    for name, fn in IMPLS.items():
        assert fn(s, list(words)) == want, (name, s, words)
    count += 1
    matches += want
    total_words += len(words)
print(count, "input ngẫu nhiên; tổng số từ", total_words, "trong đó", matches, "từ là dãy con: mọi cài đặt trùng oracle")

# tính chất: thêm một ký tự vào s không làm số khớp giảm; thêm một từ trùng tăng đúng 0 hoặc 1
rng = random.Random(5)
for _ in range(300):
    s, words = random_instance(rng, rng.randint(1, 40), rng.randint(1, 10), 5, "abc")
    base = Solution().numMatchingSubseq(s, list(words))
    ch = rng.choice("abc")
    pos = rng.randint(0, len(s))
    assert Solution().numMatchingSubseq(s[:pos] + ch + s[pos:], list(words)) >= base
    assert Solution().numMatchingSubseq(s, list(words) + [words[0]]) in (base, base + 1)
print("tính chất đơn điệu: OK")

1500 input ngẫu nhiên; tổng số từ 44795 trong đó 41561 từ là dãy con: mọi cài đặt trùng oracle
tính chất đơn điệu: OK


### F. Giới hạn lớn nhất

`|s| = 5·10^4`, `5000` từ dài tới `50`. Các trường hợp: bảng chữ đầy đủ và bảng 2 chữ cái (với `s` dài thì hầu hết từ đều khớp), `s` toàn `'a'` với từ `"a"*50` (tất cả khớp), `s` toàn `'a'` với từ có chữ `'b'` (không khớp), và một ca hỗn hợp `s` ngắn / từ dài có cả từ khớp lẫn không khớp. Đối chiếu với `count_find` (oracle nhanh) và `oracle_iter`.

In [26]:
rng = random.Random(777)
big_cases = {
    "26 chữ cái": random_instance(rng, 50000, 5000, 50, ALPHA),
    "2 chữ cái": random_instance(rng, 50000, 5000, 50, "ab", match_ratio=0.7),
    "toàn a + từ a*50": ("a" * 50000, ["a" * 50] * 5000),
    "toàn a + từ có b": ("a" * 50000, ["a" * 49 + "b"] * 5000),
    "từ dài 1": random_instance(rng, 50000, 5000, 1, ALPHA),
    "s ngắn, từ dài (hỗn hợp)": random_instance(rng, 1500, 5000, 50, ALPHA, match_ratio=0.3),
}
for name, (s, words) in big_cases.items():
    want = oracle_iter(s, words)
    t0 = time.perf_counter()
    for impl in ("find", "bisect", "bucket", "next", "Solution", "SolutionBisect", "SolutionBucket", "SolutionNext"):
        assert IMPLS[impl](s, list(words)) == want, (name, impl)
    print(f"{name:20s}: đáp án {want:5d}, 8 cài đặt đều đúng ({(time.perf_counter() - t0) * 1000:7.0f} ms tổng)")
assert oracle_iter(*big_cases["toàn a + từ a*50"]) == 5000 and oracle_iter(*big_cases["toàn a + từ có b"]) == 0
assert 0 < oracle_iter(*big_cases["s ngắn, từ dài (hỗn hợp)"]) < 5000       # ca có cả từ khớp lẫn không khớp

26 chữ cái          : đáp án  5000, 8 cài đặt đều đúng (    297 ms tổng)


2 chữ cái           : đáp án  5000, 8 cài đặt đều đúng (    287 ms tổng)


toàn a + từ a*50    : đáp án  5000, 8 cài đặt đều đúng (    379 ms tổng)


toàn a + từ có b    : đáp án     0, 8 cài đặt đều đúng (    371 ms tổng)
từ dài 1            : đáp án  5000, 8 cài đặt đều đúng (    151 ms tổng)


s ngắn, từ dài (hỗn hợp): đáp án  4945, 8 cài đặt đều đúng (    150 ms tổng)


## 13. Các lỗi người mới hay gặp

### Lỗi 1: nhầm dãy con với chuỗi con (`w in s`)

`"ace" in "abcde"` là `False` vì `"ace"` không liên tiếp, nhưng `"ace"` là dãy con. Dùng `in` cho ra đáp án thiếu.

In [27]:
s, words = ex1
print("bug_substring:", bug_substring(s, words), "| đúng:", Solution().numMatchingSubseq(s, words))
print([(w, w in s, is_subseq(w, s)) for w in words], "(chuỗi con, dãy con)")
assert bug_substring(s, words) == 1 and Solution().numMatchingSubseq(s, words) == 3

bug_substring: 1 | đúng: 3
[('a', True, True), ('bb', False, False), ('acd', False, True), ('ace', False, True)] (chuỗi con, dãy con)


**Vì sao sai:** `in` đòi các ký tự **liền nhau**. **Đúng:** kiểm tra theo thứ tự tăng dần của chỉ số (hai con trỏ / `find` với `start` / `bisect`).

### Lỗi 2: quên bắt đầu tìm **sau** vị trí vừa khớp

Nếu mỗi chữ của từ được tìm lại từ đầu `s`, thì `"bb"` tìm `b` hai lần ra cùng vị trí `1`, và ta tính `"bb"` là khớp.

In [28]:
print("bug_restart:", bug_restart(s, words), "| đúng:", Solution().numMatchingSubseq(s, words), "(đếm cả 'bb' do dùng lại chữ b@1)")
assert bug_restart(s, words) == 4

bug_restart: 4 | đúng: 3 (đếm cả 'bb' do dùng lại chữ b@1)


**Vì sao sai:** mỗi ký tự của `s` chỉ được dùng **một lần**. **Đúng:** `pos = s.find(ch, pos) + 1` (hoặc `cur = p + 1`).

### Lỗi 3: sửa danh sách trong lúc duyệt (bucket)

Nếu không tách hàng chờ ra trước, một từ vừa chuyển sang hàng chờ **cùng chữ** (như `"aa"` khi gặp `a`) lập tức bị xử lý **trong cùng vòng lặp**, tức là một ký tự `s[t]` khớp hai ký tự liên tiếp của từ.

In [29]:
s2, w2 = "a", ["aa", "a"]
print("s =", repr(s2), "words =", w2)
print("bug_bucket_same_list:", bug_bucket_same_list(s2, w2), "| đúng:", Solution().numMatchingSubseq(s2, w2), "('aa' không thể là dãy con của 'a')")
assert bug_bucket_same_list(s2, w2) == 2 and Solution().numMatchingSubseq(s2, w2) == 1

s = 'a' words = ['aa', 'a']
bug_bucket_same_list: 2 | đúng: 1 ('aa' không thể là dãy con của 'a')


**Vì sao sai:** Python cho phép nối thêm vào list đang duyệt: phần tử mới được duyệt tiếp. Từ `"aa"` được thêm vào cùng hàng chờ `a` và khớp luôn chữ `a` thứ hai bằng cùng ký tự `s[0]`. **Đúng:** `waiting = buckets.pop(ch, [])` (lấy ra rồi mới xử lý), hoặc `waiting, buckets[ch] = buckets[ch], []`.

### Lỗi 4: bỏ trùng bằng `set`

`set(words)` làm mất bội số: `["a", "a", "a"]` phải cho `3`, không phải `1`.

In [30]:
s3, w3 = "abc", ["a", "a", "ab", "ab", "ab"]
print("bug_set:", bug_set(s3, w3), "| đúng:", Solution().numMatchingSubseq(s3, w3))
# cách tối ưu đúng: kiểm tra mỗi từ khác nhau một lần rồi nhân với bội số
cnt = Counter(w3)
print("tối ưu đúng bằng Counter:", sum(is_subseq(w, s3) * m for w, m in cnt.items()))
assert bug_set(s3, w3) == 2 and Solution().numMatchingSubseq(s3, w3) == 5

bug_set: 2 | đúng: 5
tối ưu đúng bằng Counter: 5


**Vì sao sai:** đề đếm **số phần tử** của `words`, trùng cũng tính. **Đúng:** `Counter` rồi nhân bội số (nếu muốn tối ưu), hoặc đơn giản duyệt hết.

### Lỗi 5: dùng `bisect_right` thay `bisect_left`

`cur` là vị trí **được phép bắt đầu** (có thể chính là vị trí cần khớp). `bisect_left(lst, cur)` lấy phần tử `>= cur`; `bisect_right` lấy `> cur` nên bỏ qua đúng vị trí `cur` và làm mất khớp.

In [31]:
s4, w4 = "abc", ["abc", "bc", "c"]
print("bug_bisect_right_cur:", bug_bisect_right_cur(s4, w4), "| đúng:", Solution().numMatchingSubseq(s4, w4))
wrong = sum(1 for s_, w_ in tests if bug_bisect_right_cur(s_, list(w_)) != oracle_iter(s_, w_))
print("số ca sai trên", len(tests), "ca ngẫu nhiên:", wrong)
assert wrong > 0

bug_bisect_right_cur: 1 | đúng: 3
số ca sai trên 600 ca ngẫu nhiên: 187


**Vì sao sai:** khi từ bắt đầu (`cur = 0`) ký tự đầu nằm đúng ở vị trí `0`, `bisect_right(lst, 0)` bỏ qua nó. **Đúng:** `bisect_left(lst, cur)`; nhớ `cur = vị trí khớp + 1`.

### Lỗi 6 (bonus): quét hai con trỏ cho mọi từ

Đúng nhưng chậm: `5000` từ × `5·10^4` = `2,5·10^8` bước. Với ràng buộc của đề, cách này quá thời gian. Bucket/nhị phân/bảng `next` là các cách đúng bậc.

## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** "kiểm tra nhiều chuỗi có là dãy con của **một** chuỗi lớn" → **tiền xử lý chuỗi lớn** (vị trí theo chữ + nhị phân, bảng `next`) hoặc **quét chuỗi lớn một lần** với hàng chờ theo chữ (bucket).

**Mẹo:**

1. Kiểm tra dãy con luôn là **tham lam khớp sớm nhất**.
2. Nhiều từ, một `s` lớn → đừng quét `s` cho từng từ. Hỏi: "tôi cần *lần xuất hiện kế tiếp của chữ `c` từ vị trí `p`* — làm sao tra nhanh?"
3. Bucket: **tách hàng chờ ra trước** khi xử lý để mỗi ký tự của `s` chỉ khớp một ký tự của mỗi từ.
4. Bảng `next[c][i]`: dựng từ phải sang trái; nhớ cột lính canh `n` bằng `-1`.
5. Từ trùng: dùng `Counter` rồi nhân bội số, đừng dùng `set` rồi quên nhân.
6. Có thể dùng **Trie** các từ rồi quét `s` với tập nút hiện tại (ý tưởng bucket mở rộng cho tiền tố chung).

**Họ bài cùng pattern:** Is Subsequence (392), Shortest Way to Form String (1055), Longest Word in Dictionary through Deleting (524), Append Characters to String to Make Subsequence (2486), Distinct Subsequences (115).

## 15. Biến thể

1. **Một từ, nhiều `s`** (Is Subsequence, follow-up 392): tiền xử lý từng `s` (hoặc dùng bảng `next`).
2. **Số lần ghép `s` để tạo thành target** (1055): mỗi lần dùng `next` trên một bản `s`, hết thì sang bản kế.
3. **Từ dài nhất trong từ điển là dãy con của `s`** (524): kiểm tra từng từ rồi chọn.
4. **Trả về danh sách các từ khớp** thay vì số lượng.
5. **Vị trí kết thúc khớp** của từng từ (bài tập 5).

Cài đặt biến thể 1–2 và đối chiếu:

In [32]:
def is_subsequence_many(s_list, t):
    """Biến thể 1: kiểm tra một từ t với nhiều chuỗi s bằng bảng next (mỗi s dựng bảng một lần)."""
    out = []
    for s in s_list:
        nxt = build_next(s)
        cur = 0
        ok = True
        for ch in t:
            p = nxt[ord(ch) - 97][cur]
            if p == -1:
                ok = False
                break
            cur = p + 1
        out.append(ok)
    return out


print(is_subsequence_many(["abcde", "aec", "xaybzc"], "abc"))
assert is_subsequence_many(["abcde", "aec", "xaybzc"], "abc") == [True, False, True]


def shortest_way(source, target):
    """Biến thể 2 (LeetCode 1055): số bản source ít nhất để ghép (làm dãy con) thành target, hoặc -1."""
    nxt = build_next(source)
    copies, cur = 1, 0
    for ch in target:
        p = nxt[ord(ch) - 97][cur]
        if p == -1:
            copies += 1
            cur = 0
            p = nxt[ord(ch) - 97][0]
            if p == -1:                        # chữ này không có trong source
                return -1
        cur = p + 1
    return copies


print(shortest_way("abc", "abcbc"), shortest_way("abc", "acdbc"), shortest_way("xyz", "xzyxz"))
assert shortest_way("abc", "abcbc") == 2 and shortest_way("abc", "acdbc") == -1 and shortest_way("xyz", "xzyxz") == 3

[True, False, True]
2 -1 3


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Hai con trỏ mỗi từ | Quét `s` cho từng từ | `O(m · n)` | `O(1)` | Oracle; `n, m` nhỏ |
| 2. Vị trí + `bisect_left` | Nhảy tới lần xuất hiện kế tiếp bằng nhị phân | `O(n + W log n)` | `O(n)` | Đơn giản, nhanh, dễ viết |
| 3. Bucket | Quét `s` một lần, từ chờ theo chữ cái | `O(n + W)` | `O(m)` | **Đáp án chính**; gọn bộ nhớ |
| 4. Bảng `next[c][i]` | Tra `O(1)` lần xuất hiện kế tiếp | `O(26n + W)` | `O(26n)` | Rất nhiều truy vấn trên cùng `s` |

(`m` = số từ, `n = |s|`, `W` = tổng độ dài các từ.) Cả bốn khớp nhau trên 1092 chuỗi vét cạn × 46 từ và 1500 input ngẫu nhiên (mục 12).

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Đoán trước rồi mới chạy:

In [33]:
s = "abcab"
print([w for w in ("abab", "aab", "bca", "ba", "cc", "bb") if is_subseq(w, s)])
print(s.find("a", 1), s.find("c", 3), s.find("b", 4))

['abab', 'aab', 'bca', 'ba', 'bb']
3 -1 4


### Bài 2 (tính tay)

`s = "dsahjpjauf"`, `words = ["ahjpjau", "ja", "ahbwzgqnuk", "tnmlanowax"]`.

- (a) Với `"ahjpjau"`, ghi lại vị trí khớp tham lam từng chữ.
- (b) Với `"ja"` vì sao khớp (hai chữ `j` ở đâu)?
- (c) Với `"ahbwzgqnuk"`, đến chữ nào thì thất bại?

Tự giải rồi kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện cách danh sách vị trí + `bisect_left`:

In [34]:
def count_todo(s, words):
    pos = defaultdict(list)
    for i, ch in enumerate(s):
        pos[ch].append(i)
    ans = 0
    for w in words:
        cur = 0
        ok = True
        for ch in w:
            lst = pos.get(ch, [])
            # TODO 1: k = chỉ số đầu tiên của lst có giá trị >= cur (bisect_left)
            # TODO 2: nếu k == len(lst) thì ok = False và dừng; ngược lại cur = lst[k] + 1
            pass
        if ok:
            ans += 1
    return ans


# Khi làm xong, bỏ comment để kiểm tra:
# assert count_todo("abcde", ["a", "bb", "acd", "ace"]) == 3

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết lớp `MySolution` bằng bucket (hàng chờ theo chữ cái):

In [35]:
class MySolution:
    def numMatchingSubseq(self, s, words):
        pass            # TODO


# Khi làm xong:
# assert MySolution().numMatchingSubseq("abcde", ["a", "bb", "acd", "ace"]) == 3
# assert MySolution().numMatchingSubseq("a", ["aa", "a"]) == 1

### Bài 5 (mini challenge): vị trí kết thúc khớp

Viết `match_end_positions(s, words)` trả về danh sách, với mỗi từ là **chỉ số trong `s` của ký tự cuối cùng được khớp tham lam**, hoặc `-1` nếu từ không phải dãy con. Làm trong `O(|s| + Σ|w|)` (một trong hai cách: bucket hoặc bảng `next`).

In [36]:
def match_end_positions(s, words):
    pass            # TODO


# Khi làm xong:
# assert match_end_positions("abcde", ["a", "bb", "acd", "ace"]) == [0, -1, 3, 4]

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: kiểm tra từng từ theo hai con trỏ.
- Mức 2: `s = "abcab"`: `"abab"` có (`a0 b1 a3 b4`), `"aab"` có (`a0 a3 b4`), `"bca"` có (`b1 c2 a3`).
- Mức 3: danh sách `['abab', 'aab', 'bca', 'ba', 'bb']` (`"bb"`: `b@1`, `b@4`; `"cc"` thì không vì chỉ có một `c`); `find`: `3, -1, 4` (sau vị trí 3 không còn `c`).

**Bài 2**

- Mức 1: tham lam khớp sớm nhất.
- Mức 2: `ahjpjau` trên `dsahjpjauf`: `a@2, h@3, j@4, p@5, j@6, a@7, u@8`.
- Mức 3: `ja`: `j@4`, rồi `a@7`; `ahbwzgqnuk`: `a@2, h@3, b` không còn → thất bại ở chữ `b`.

**Bài 3**

- Mức 1: `from bisect import bisect_left`.
- Mức 2: `k = bisect_left(lst, cur)`.
- Mức 3: `if k == len(lst): ok = False; break` rồi `cur = lst[k] + 1`.

**Bài 4**

- Mức 1: `buckets[chữ đầu]` chứa `(từ, chỉ số)`.
- Mức 2: với mỗi `ch` trong `s`: `waiting = buckets.pop(ch, [])`.
- Mức 3: xem mục 9.

**Bài 5**

- Mức 1: dùng bảng `next` hoặc `find`.
- Mức 2: cuối cùng `cur - 1` là vị trí chữ cuối được khớp.
- Mức 3: nếu thất bại tại chữ nào thì trả `-1`.

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert` để chạy kiểm tra.

### Bài 1

In [37]:
s = "abcab"
assert [w for w in ("abab", "aab", "bca", "ba", "cc", "bb") if is_subseq(w, s)] == ["abab", "aab", "bca", "ba", "bb"]
assert (s.find("a", 1), s.find("c", 3), s.find("b", 4)) == (3, -1, 4)
print("Bài 1 OK")

Bài 1 OK


### Bài 2

(a) `a@2, h@3, j@4, p@5, j@6, a@7, u@8`. (b) `"ja"`: `j@4` rồi `a@7` (chữ `a` đầu tiên sau vị trí 4). (c) `"ahbwzgqnuk"`: `a@2, h@3`, rồi cần `b` sau vị trí 3: `s = "dsahjpjauf"` không có `b` → thất bại ở chữ `b`.

In [38]:
s = "dsahjpjauf"
def greedy_positions(s, w):
    out, cur = [], 0
    for ch in w:
        p = s.find(ch, cur)
        if p == -1:
            return out, ch
        out.append(p)
        cur = p + 1
    return out, None


assert greedy_positions(s, "ahjpjau") == ([2, 3, 4, 5, 6, 7, 8], None)
assert greedy_positions(s, "ja") == ([4, 7], None)
assert greedy_positions(s, "ahbwzgqnuk") == ([2, 3], "b")
assert Solution().numMatchingSubseq(s, ["ahjpjau", "ja", "ahbwzgqnuk", "tnmlanowax"]) == 2
print("Bài 2 OK")

Bài 2 OK


### Bài 3

In [39]:
def count_answer(s, words):
    pos = defaultdict(list)
    for i, ch in enumerate(s):
        pos[ch].append(i)
    ans = 0
    for w in words:
        cur = 0
        ok = True
        for ch in w:
            lst = pos.get(ch, [])
            k = bisect_left(lst, cur)
            if k == len(lst):
                ok = False
                break
            cur = lst[k] + 1
        if ok:
            ans += 1
    return ans


assert count_answer("abcde", ["a", "bb", "acd", "ace"]) == 3
rng = random.Random(3)
for _ in range(200):
    s_, w_ = random_instance(rng, rng.randint(1, 30), rng.randint(1, 8), 5, "abc")
    assert count_answer(s_, w_) == oracle_iter(s_, w_)
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [40]:
class MySolutionAnswer:
    def numMatchingSubseq(self, s, words):
        buckets = defaultdict(list)
        for w in words:
            buckets[w[0]].append((w, 0))
        ans = 0
        for ch in s:
            waiting = buckets.pop(ch, [])
            for w, i in waiting:
                if i + 1 == len(w):
                    ans += 1
                else:
                    buckets[w[i + 1]].append((w, i + 1))
        return ans


assert MySolutionAnswer().numMatchingSubseq("abcde", ["a", "bb", "acd", "ace"]) == 3
assert MySolutionAnswer().numMatchingSubseq("a", ["aa", "a"]) == 1
rng = random.Random(4)
for _ in range(300):
    s_, w_ = random_instance(rng, rng.randint(1, 40), rng.randint(1, 10), 6, "ab")
    assert MySolutionAnswer().numMatchingSubseq(s_, w_) == oracle_iter(s_, w_)
print("Bài 4 OK")

Bài 4 OK


### Bài 5

Với bảng `next`: sau khi khớp hết, vị trí kết thúc là `cur − 1`.

In [41]:
def match_end_positions_answer(s, words):
    nxt = build_next(s)
    out = []
    for w in words:
        cur, ok = 0, True
        for ch in w:
            p = nxt[ord(ch) - 97][cur]
            if p == -1:
                ok = False
                break
            cur = p + 1
        out.append(cur - 1 if ok else -1)
    return out


def match_end_oracle(s, words):
    res = []
    for w in words:
        j = last = 0
        pos = -1
        for i, ch in enumerate(s):
            if j < len(w) and ch == w[j]:
                j += 1
                pos = i
        res.append(pos if j == len(w) else -1)
    return res


assert match_end_positions_answer("abcde", ["a", "bb", "acd", "ace"]) == [0, -1, 3, 4]
rng = random.Random(5)
for _ in range(300):
    s_, w_ = random_instance(rng, rng.randint(1, 40), rng.randint(1, 8), 6, "abc")
    assert match_end_positions_answer(s_, w_) == match_end_oracle(s_, w_)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (đếm nhiều từ là dãy con của `s`, bucket):**

```python
class Solution:
    def numMatchingSubseq(self, s, words):
        buckets = defaultdict(list)
        for w in words:
            buckets[w[0]].append((w, 0))
        ans = 0
        for ch in s:
            waiting = buckets.pop(ch, [])         # tách ra TRƯỚC khi xử lý
            for w, i in waiting:
                if i + 1 == len(w):
                    ans += 1
                else:
                    buckets[w[i + 1]].append((w, i + 1))
        return ans
```

**Template (nhảy bằng nhị phân):** `cur = 0; for ch in w: k = bisect_left(pos[ch], cur); (thất bại nếu k == len) ; cur = pos[ch][k] + 1`.

**Checklist trước khi submit:**

- [ ] Hiểu rõ **dãy con** (không cần liền nhau), không dùng `w in s`?
- [ ] Mỗi ký tự của từ tìm **sau** vị trí vừa khớp (`+ 1`), mỗi ký tự của `s` chỉ dùng một lần?
- [ ] Bucket: tách hàng chờ ra (`pop`) trước khi xử lý?
- [ ] Từ trùng được đếm **đủ bội số** (không dùng `set` rồi quên nhân)?
- [ ] Dùng `bisect_left` (không phải `bisect_right`) với `cur` là vị trí được phép bắt đầu?
- [ ] Độ phức tạp `O(|s| + Σ|w|)` hoặc `O(|s| + Σ|w| log |s|)`, không `O(m · |s|)`?
- [ ] Đúng tên `class Solution`, `numMatchingSubseq(self, s, words)`, không in debug.